# MSR RQ1: which rules is each agent PR under?

1. Load agent PRs (AIDev) and context file history (GHAgentFiles)
2. For each PR, find the context files that existed when it was opened
3. Detect which rules (from `rules.md`) those files state
4. Count PRs per rule

In [1]:
import re
from functools import cache

import pandas as pd

DATA = "../../instruction-data"
AIDEV = "hf://datasets/hao-li/AIDev@v4"

## 1. Load data

In [2]:
prs = pd.read_parquet(
    f"{AIDEV}/pull_request.parquet",
    columns=["id", "agent", "title", "state", "created_at", "merged_at", "repo_url", "html_url"],
)
prs["repo"] = prs.repo_url.str.replace("https://api.github.com/repos/", "").str.lower()
prs["created_at"] = pd.to_datetime(prs.created_at, utc=True)

changes = pd.read_csv(f"{DATA}/context_metadata.csv", low_memory=False)
changes["repo"] = changes.repository.str.lower()
changes["date"] = pd.to_datetime(changes.commit_date, unit="s", utc=True)
changes = changes[~changes.symbolic_link]  # symlinks just point to another context file

changes_by_repo = {repo: group for repo, group in changes.groupby("repo")}

print(len(prs), "PRs,", len(changes), "context file changes")

71677 PRs, 235188 context file changes


## 2. Context files that existed when each PR was opened

In [3]:
def context_files_at(repo, when):
    """Hashes of the latest version of each context file in `repo` committed before `when`."""
    if repo not in changes_by_repo:
        return []

    history = changes_by_repo[repo]
    before = history[history.date < when]

    latest = before.sort_values("date").groupby("file_path").tail(1)
    latest = latest[latest.git_change_type != "D"]

    # a renamed file still shows its old path as alive -> drop old paths
    renamed_away = before.loc[before.git_change_type == "R", "previous_file_path"]
    latest = latest[~latest.file_path.isin(renamed_away)]

    return list(latest.file_hash.dropna())  # some renames have no stored content


prs["context_hashes"] = [context_files_at(repo, when) for repo, when in zip(prs.repo, prs.created_at)]

prs = prs[prs.context_hashes.str.len() > 0]
print(len(prs), "PRs in", prs.repo.nunique(), "repos had a context file")

23422 PRs in 1326 repos had a context file


## 3. Which rules do those files state?

Simple keyword regexes per rule (see `rules.md`). They will have false positives; we validate by hand later.

In [4]:
MANAGER = r"`?(pnpm|yarn|bun|uv|poetry)`?(?:\s*\([^)]*\))?"  # optional "(v9+)" after the name
PACKAGE_MANAGER_PHRASES = [
    rf"\b(?:use|using|prefer)\s+{MANAGER}\s+(?:as|for)\s+(?:the\s+|our\s+|your\s+)?(?:package|dependenc|js\b|javascript|node|python)",  # use pnpm as the package manager
    rf"\b(?:use|using)\s+{MANAGER}\s+(?:only|exclusively)\b",  # use yarn exclusively
    rf"\bonly\s+use\s+{MANAGER}",  # only use bun
    rf"\b(?:use|run)\s+`?(pnpm|yarn|bun|uv|poetry)\s+(?:install|add|sync)\b",  # use `pnpm install`
    rf"package\s+manager\b[\s*:`_-]*(?:is\s+|use\s+)?{MANAGER}",  # **Package manager**: uv
    rf"package\s+manager,\s+use\s+{MANAGER}",  # ...need a package manager, use pnpm
    rf"{MANAGER}[\s*:_-]+package\s+manager",  # **PNPM**: Package manager
    rf"{MANAGER}\s+is\s+(?:the|our)\b[^.\n]{{0,20}}package\s+manager",  # pnpm is the project's package manager
]

# `^(?!.*X)` means "the line does not contain X"
RULE_PATTERNS = {
    "commit_format": r"^(?!.*\b(do not|don't|never|consider|encouraged|optional|when possible|if possible|when reasonable)\b).*conventional commits?",
    "required_tests": (
        r"\b(must|should|always|required?)\b[^.\n]*\b(add|write|include|accompanied)\b[^.\n]{0,30}\btests?\b"
        r"|\b(add|write|include)\b[^.\n]{0,30}\btests?\b[^.\n]*\b(whenever|for (any|every|all|new))\b"
    ),
    "package_manager": "|".join(PACKAGE_MANAGER_PHRASES),
    "changelog": r"^(?!.*\brelease).*\b(update|add|edit)\b[^.\n]{0,40}\bchangelog\b",
}


@cache
def read_file(file_hash):
    with open(f"{DATA}/context_content/{file_hash}", encoding="utf-8", errors="replace") as f:
        return f.read()


def context_text(hashes):
    return "\n".join(read_file(h) for h in hashes)


# Lines that match a pattern but are not a rule for every PR (found in hand validation)
RELEASE_HEADING = r"\b(releas\w*|publish\w*|deploy\w*|version bump)\b"
NOT_A_RULE = (
    r"\b(if the user|if applicable|where applicable|when applicable|if needed|if necessary|when appropriate"
    r"|if possible|consider|for doc(umentation)? changes)\b"
    r"|same directory|next to|alongside|\bnam(ed|ing)\b|\blocated\b"  # where tests go, not that they are required
    r"|\b(pr|pull request) (description|template|body)\b"
    r"|/changelog/"  # link to a changelog page, e.g. "[Update Indicator](/changelog/2024-06-10-...)"  # changelog section of the PR text, not a file
)
FILTERED_RULES = {"required_tests", "changelog"}


@cache
def rule_units(hashes, rule):
    """Matching lines of `rule` (joined with the lines they continue), minus release steps and conditional lines."""
    units, heading, previous = [], "", ""
    for line in context_text(hashes).splitlines():
        stripped = line.strip()
        if stripped.startswith("#"):
            heading = stripped
        unit = f"{previous} {stripped}" if stripped[:1].islower() else stripped  # wrapped sentence
        previous = unit if stripped else ""
        if not re.search(RULE_PATTERNS[rule], stripped, re.IGNORECASE):
            continue
        if rule in FILTERED_RULES and (
            re.search(RELEASE_HEADING, heading, re.IGNORECASE) or re.search(NOT_A_RULE, unit, re.IGNORECASE)
        ):
            continue
        units.append(unit)
    return tuple(units)


def stated_rules(hashes):
    hashes = tuple(hashes)
    return [rule for rule in RULE_PATTERNS if rule_units(hashes, rule)]


prs["rules"] = prs.context_hashes.apply(stated_rules)

## 4. How many PRs fall under each rule?

In [5]:
pr_rule = prs.explode("rules", ignore_index=True).dropna(subset=["rules"])

pr_rule.groupby("rules").agg(prs=("id", "count"), repos=("repo", "nunique"))

,prs,repos
rules,,
changelog,917,34
commit_format,2088,144
package_manager,2002,128
required_tests,2411,173


In [6]:
pd.crosstab(pr_rule.rules, pr_rule.agent)

agent,Claude_Code,Copilot,Cursor,Devin,Google_Jules,OpenAI_Codex
rules,,,,,,
changelog,25,371,79,44,15,383
commit_format,210,778,276,184,0,640
package_manager,123,576,708,332,2,261
required_tests,213,1506,268,67,1,356


## 5. Sanity check: do the regexes catch real rules?

Print random lines that matched each rule. If many are not real rules, tighten the pattern in step 3 and rerun.

In [7]:
def matching_lines(rule, n=20):
    hashes = pr_rule.loc[pr_rule.rules == rule, "context_hashes"].explode().unique()

    lines = []
    for file_hash in hashes:
        for line in read_file(file_hash).splitlines():
            if re.search(RULE_PATTERNS[rule], line, re.IGNORECASE):
                lines.append(line.strip())

    lines = pd.Series(lines).drop_duplicates()
    return lines.sample(min(n, len(lines)), random_state=0)


for rule in RULE_PATTERNS:
    print(f"\n===== {rule} =====")
    for line in matching_lines(rule):
        print("-", line[:200])


===== commit_format =====
- - Commitizen for conventional commits
- - **Conventional Commits**: `feat:`, `fix:`, `docs:`, `refactor:`, `chore:`
- - Use Conventional Commits where possible: `feat:`, `fix:`, `chore:`, `refactor:`, etc. Keep messages imperative and scoped (e.g., `feat(api): add chat stream`).
- - Conventional commits enforced by hook: `type(scope): subject` ≤ 50 chars; types: `feat|fix|docs|dx|style|refactor|perf|test|workflow|build|ci|chore|types|wip|release`.
- - Ensure the PR title is descriptive and follows the conventional commit format
- - Use Conventional Commits: `type(scope): description` in imperative mood, ≤72 characters, no trailing period.
- - Follow the [Conventional Commits](https://www.conventionalcommits.org/) specification for all commit messages
- - **Commit Messages**: Use clear, descriptive commit messages following the Conventional Commits format:
- - Follow [Conventional Commits 1.0.0](https://www.conventionalcommits.org/en/v1.0.0/)
- - Use convent

- - Always add test fixtures when modifying detection patterns
- - **Always add tests for new features**
- **Every new feature MUST include integration tests that validate ACTUAL platform behavior.** Integration tests are not optional - they are required because they:
- 2. **Write tests first** for new components and hooks before implementation
- 3. Add/update tests for new features
- - ALL new features MUST include comprehensive unit tests
- 5. Write tests for new functionality
- - Write go unit tests whenever possible
- **MANDATORY REQUIREMENT:** Every code review must include a test suggestion using the Quick Reference Template provided below. This is required for all pull request reviews to ensure consistent test c
- Stream indicators must include performance tests in the `tests/performance` project for high-frequency scenarios:
- - Add tests for new API endpoints and features
- - Always include test cases for critical paths of the application.
- - Write unit tests for new function

- Run `pnpm install` to hydrate the workspace (Node.js ≥ 20.19.0 is required). Build all packages with `pnpm run build`. Execute `pnpm run test` for the Vitest suite or `pnpm run test:cov` to generate c
- **Package Manager**: Use Yarn exclusively (configured in .yarnrc.yml)
- - Use `pnpm add` instead of `npm install <package>`
- 5. Run `pnpm install` to update dependencies
- - Use Yarn (not npm) for package management
- - do not apply any migrations, just use `pnpm sync`
- - Use `pnpm` as the package manager
- - Use `pnpm add` for adding dependencies instead of `npm install` or `yarn add`
- ## Use pnpm Only
- - **Use Bun exclusively.** Run installs with `bun install`, scripts with `bun run <script>`, add deps via `bun add` / `bun add -d`.
- - **Package Manager**: pnpm (v10+)
- - [uv.lock](mdc:uv.lock) - Locked dependencies using uv package manager
- - Yarn package manager (v1.22.19+)
- Only use pnpm
- - When suggesting dependency management, use `uv sync` instead of `pip install -r req

## 6. Which files did each PR change?

`pr_commit_details` has one row per (commit, changed file). We keep the file names and which ones were newly added. `pr_commits` gives the commit messages.

In [8]:
files = pd.read_parquet(f"{AIDEV}/pr_commit_details.parquet", columns=["pr_id", "filename", "status"])
files = files[files.pr_id.isin(prs.id)].dropna()

changed_files = files.groupby("pr_id").filename.apply(set)
prs["changed_files"] = prs.id.map(changed_files)

added_files = files[files.status == "added"].groupby("pr_id").filename.apply(set)
prs["added_files"] = [added_files.get(pr_id, set()) for pr_id in prs.id]

commits = pd.read_parquet(f"{AIDEV}/pr_commits.parquet", columns=["pr_id", "message"])
commits = commits[commits.pr_id.isin(prs.id)].dropna()
commit_messages = commits.groupby("pr_id").message.apply(lambda msgs: [m.split("\n")[0] for m in msgs])
prs["commit_messages"] = [commit_messages.get(pr_id, []) for pr_id in prs.id]

print(prs.changed_files.isna().sum(), "of", len(prs), "PRs have no file info")
prs[["title", "changed_files"]].head()

590 of 23422 PRs have no file info


,title,changed_files
3,Use permanent cache for library manga covers,{app/src/main/java/eu/kanade/tachiyomi/util/Ma...
4,Optimize LibraryComposePresenter Performance,{app/src/main/java/eu/kanade/tachiyomi/data/do...
13,Finalize and Test `optiland.environment` Subpa...,"{tests/test_sensitivity_analysis.py, docs/api/..."
15,Pull request for issue #324,"{scripts/download-uv.mjs, scripts/before-pack...."
17,feat: Add video tips to permission onboarding,{app/src/main/res/layout/activity_onboarding_s...


## 7. Checkers

Each checker gets one (PR, rule) row and returns `not_applicable`, `complied` or `violated`.

### Commit / PR title format

Complied if the **PR title or any commit message** follows Conventional Commits.
A leading `[tag]` is ignored (some repos ask for e.g. `[alpha_factory] feat: ...`).
This is lenient on purpose: we would rather miss a violation than invent one.

In [9]:
# "type(scope)!: description" - the spec allows any type word (repos add their own, e.g. "k8s:")
CONVENTIONAL_TITLE = r"^[\w,-]+(\(.+\))?!?: .+"


def follows_format(text):
    text = re.sub(r"^\[[^\]]*\]\s*", "", text.strip())
    text = re.sub(r"^[^\w(\[]+", "", text)  # emoji conventional commits: "🐛 fix: ..."
    return re.match(CONVENTIONAL_TITLE, text, re.IGNORECASE) is not None


def check_commit_format(pr):
    if follows_format(pr.title) or any(follows_format(m) for m in pr.commit_messages):
        return "complied"
    return "violated"


(
    check_commit_format(pd.Series({"title": "[alpha_factory] add x", "commit_messages": ["feat: add x"]})),  # complied
    check_commit_format(pd.Series({"title": "Add sidebar", "commit_messages": ["Initial plan"]})),  # violated
)

('complied', 'violated')

### Package manager

1. Read which manager(s) the context file asks for (e.g. `pnpm`).
2. The rule applies only if the PR changed a lockfile of the same ecosystem (JS or Python).
3. Violated if the PR **added a new** lockfile of a different manager (e.g. a new `package-lock.json` when pnpm is required).
   Editing a wrong lockfile that already existed is not counted: some sub-packages legitimately use another manager.
4. Lockfiles inside `test/`, `fixtures/`, `examples/` folders are ignored (sample projects).

In [10]:
LOCKFILE_MANAGER = {
    "package-lock.json": "npm",
    "pnpm-lock.yaml": "pnpm",
    "yarn.lock": "yarn",
    "bun.lock": "bun",
    "bun.lockb": "bun",
    "uv.lock": "uv",
    "poetry.lock": "poetry",
}
ECOSYSTEM = {"npm": "js", "pnpm": "js", "yarn": "js", "bun": "js", "uv": "python", "poetry": "python"}


IGNORED_FOLDERS = {"test", "tests", "fixtures", "__fixtures__", "example", "examples"}


def required_managers(text):
    found = set()
    for phrase in PACKAGE_MANAGER_PHRASES:
        found |= {name.lower() for name in re.findall(phrase, text, re.IGNORECASE)}
    return found


def in_ignored_folder(path):
    folders = path.lower().split("/")[:-1]
    return any(folder in IGNORED_FOLDERS for folder in folders)


def lockfile_managers(file_paths, ecosystems):
    managers = {LOCKFILE_MANAGER.get(path.split("/")[-1]) for path in file_paths if not in_ignored_folder(path)}
    return {m for m in managers if m and ECOSYSTEM[m] in ecosystems}


def check_package_manager(pr):
    required = required_managers(context_text(pr.context_hashes))
    ecosystems = {ECOSYSTEM[m] for m in required}

    changed = lockfile_managers(pr.changed_files, ecosystems)
    added = lockfile_managers(pr.added_files, ecosystems)

    if added - required:
        return "violated"
    if changed and changed <= required:
        return "complied"
    return "not_applicable"


(
    required_managers("Use pnpm as the package manager. Run `uv sync` first."),  # -> {'pnpm', 'uv'}
    required_managers("Use `yarn validate` before pushing"),  # -> set(): just a command
)

({'pnpm', 'uv'}, set())

### Required tests

1. Applies if the PR changed at least one **source file** (code that is not a test).
2. Complied if it also changed at least one **test file**, otherwise violated.

In [11]:
SOURCE_EXTENSIONS = {
    ".py", ".js", ".jsx", ".ts", ".tsx", ".go", ".rs", ".java", ".kt", ".rb", ".php",
    ".cs", ".c", ".cpp", ".h", ".swift", ".scala", ".dart", ".vue", ".svelte",
}

TEST_PATH = (
    r"(?i:(^|/)([\w-]+\.)*(unit|integration|e2e|functional)?(tests?|__tests__|specs?)/)"  # test folder: tests/, src/test/, Build.UnitTests/
    r"|(^|/)test_[^/]*$"  # test_ prefix:            test_foo.py
    r"|(?i:[._-](test|spec)s?\.[a-z]+$)"  # test/spec suffix:        foo_test.go, foo.test.ts, Foo_Tests.cs
    r"|[a-z0-9]Tests?\.[a-z]+$"  # CamelCase suffix:        FooTest.java, FooTests.cs
)


LOCKFILES = r"(^|/)([^/]+\.lock|package-lock\.json|pnpm-lock\.yaml|bun\.lockb)$"


def is_test_file(path):
    return re.search(TEST_PATH, path) is not None and re.search(LOCKFILES, path) is None


def is_source_file(path):
    extension = "." + path.split(".")[-1].lower() if "." in path else ""
    return extension in SOURCE_EXTENSIONS and not is_test_file(path)


CONVENTIONAL_KIND = {"feat": "feature", "fix": "fix", "docs": "docs"}  # other types (refactor, chore, ...) -> "other"
DOCS_WORDS = r"\b(docs?|docstrings?|jsdoc|documentation|typos?|readme)\b"
FIRST_WORD_KIND = {
    "fix": r"fix\w*|resolve\w*|bug\w*|correct\w*|patch\w*|prevent\w*|handle\w*",
    "other": r"refactor\w*|renam\w*|move\w*|clean\w*|remov\w*|delet\w*|bump\w*|upgrad\w*|updat\w*|chore|revert\w*"
             r"|migrat\w*|convert\w*|format\w*|lint\w*|optimi[sz]\w*|improv\w*|simplif\w*",
    "feature": r"add\w*|implement\w*|support\w*|introduc\w*|creat\w*|enabl\w*|allow\w*|new|feat\w*|build\w*",
}


def pr_kind(title):
    """feature / fix / docs / other / unknown, from the PR title only."""
    title = re.sub(r"^\[[^\]]*\]\s*|^[^\w(\[]+", "", title.strip()).lower()
    conventional = re.match(r"^(\w+)(\(.+?\))?!?:", title)
    if conventional:
        return CONVENTIONAL_KIND.get(conventional.group(1), "other")
    if re.search(DOCS_WORDS, title) or title.startswith("document"):
        return "docs"
    for kind, words in FIRST_WORD_KIND.items():
        if re.match(rf"({words})\b", title):
            return kind
    return "unknown"


LANGUAGE_EXTENSIONS = {
    "python": {".py"}, "typescript": {".ts", ".tsx"}, "javascript": {".js", ".jsx"}, "rust": {".rs"},
    "java": {".java"}, "kotlin": {".kt"}, "ruby": {".rb"}, "php": {".php"}, "swift": {".swift"},
}


def applies_to(unit, kind, changed_files):
    """Does one rule sentence cover this PR? Lenient: unclear cases do not apply."""
    languages = [lang for lang in LANGUAGE_EXTENSIONS if re.search(rf"\b{lang}\b", unit, re.IGNORECASE)]
    if languages:
        extensions = set().union(*(LANGUAGE_EXTENSIONS[lang] for lang in languages))
        if not any(("." + f.rsplit(".", 1)[-1].lower()) in extensions for f in changed_files if "." in f):
            return False
    if kind == "docs":
        return False
    only_new = re.search(r"\bnew\s+(\w+\s+)?(features?|functionality|code|endpoints?|algorithms?|apis?|modules?|components?|functions?|logic)\b|\bfeatures?\b|\bfunctionality\b|\bsignificant\b|\bnotable\b|user[- ]facing", unit, re.IGNORECASE)
    covers_fixes = re.search(r"\bbugs?\b|\bfix", unit, re.IGNORECASE)
    if only_new:
        return kind == "feature" or (kind == "fix" and covers_fixes is not None)
    return True


def rule_applies(pr, rule, units=None):
    units = rule_units(tuple(pr.context_hashes), rule) if units is None else units
    kind = pr_kind(pr.title)
    return any(applies_to(unit, kind, pr.changed_files) for unit in units)


def check_required_tests(pr, units=None):
    if not any(is_source_file(f) for f in pr.changed_files):
        return "not_applicable"
    if not rule_applies(pr, "required_tests", units):
        return "not_applicable"
    if any(is_test_file(f) for f in pr.changed_files):
        return "complied"
    return "violated"


[(f, is_test_file(f), is_source_file(f)) for f in ["src/app.py", "tests/test_app.py", "pkg/x_test.go", "src/FooTest.java", "README.md"]]

[('src/app.py', False, True),
 ('tests/test_app.py', True, False),
 ('pkg/x_test.go', True, False),
 ('src/FooTest.java', True, False),
 ('README.md', False, False)]

### Changelog

1. Applies if the PR changed at least one source file.
2. Complied if it changed a changelog file (`CHANGELOG.md`, `docs/changelog/...`) or a changeset (`.changeset/*.md`), otherwise violated.

In [12]:
CHANGELOG_PATH = r"changelog|(^|/)\.changeset/|(^|/)changes\.md$"


def only_when_file_changes(unit, changed_files):
    """'Whenever `foo.py` is modified, ... changelog' -> applies only if foo.py changed."""
    if not re.search(r"\b(whenever|when|if)\b", unit, re.IGNORECASE):
        return True
    files = re.findall(r"`([\w./-]+\.\w+)`", unit)
    files = [f for f in files if not re.search(CHANGELOG_PATH, f, re.IGNORECASE)]
    return not files or any(path.endswith(f) for f in files for path in changed_files)


def check_changelog(pr, units=None):
    if not any(is_source_file(f) for f in pr.changed_files):
        return "not_applicable"
    units = rule_units(tuple(pr.context_hashes), "changelog") if units is None else units
    units = [u for u in units if only_when_file_changes(u, pr.changed_files)]
    if not rule_applies(pr, "changelog", units):
        return "not_applicable"
    changelog_path = CHANGELOG_PATH + ("|(^|/)readme\\.md$" if any(re.search(r"readme", u, re.IGNORECASE) for u in units) else "")
    if any(re.search(changelog_path, f, re.IGNORECASE) for f in pr.changed_files):
        return "complied"
    return "violated"

### Label every (PR, rule) pair

In [13]:
CHECKERS = {
    "commit_format": check_commit_format,
    "package_manager": check_package_manager,
    "required_tests": check_required_tests,
    "changelog": check_changelog,
}

pr_rule = prs.explode("rules", ignore_index=True).dropna(subset=["rules"])
pr_rule = pr_rule[pr_rule.rules.isin(CHECKERS) & pr_rule.changed_files.notna()]

pr_rule["label"] = [CHECKERS[row.rules](row) for row in pr_rule.itertuples()]
pd.crosstab(pr_rule.rules, pr_rule.label)

label,complied,not_applicable,violated
rules,,,
changelog,321,381,193
commit_format,1793,0,250
package_manager,244,1712,23
required_tests,785,1287,265


## 8. RQ1: violation rate

Violation rate = violated / (complied + violated). `not_applicable` rows are left out.

`repo_averaged_rate` = mean of each repo's own rate, so a repo with 3,000 PRs counts the same as one with 10.

In [14]:
checked = pr_rule[pr_rule.label != "not_applicable"].copy()
checked["violated"] = checked.label == "violated"

summary = checked.groupby("rules").agg(
    prs=("id", "count"),
    repos=("repo", "nunique"),
    violation_rate=("violated", "mean"),
)

per_repo_rate = checked.groupby(["rules", "repo"]).violated.mean()
summary["repo_averaged_rate"] = per_repo_rate.groupby("rules").mean()
summary.round(2)

,prs,repos,violation_rate,repo_averaged_rate
rules,,,,
changelog,514,31,0.38,0.50
commit_format,2043,139,0.12,0.20
package_manager,267,61,0.09,0.18
required_tests,1050,125,0.25,0.26


In [15]:
per_agent_repo = checked.groupby(["rules", "agent", "repo"]).violated.mean()
by_agent = pd.DataFrame({
    "repo_averaged_rate": per_agent_repo.groupby(["rules", "agent"]).mean(),
    "prs": checked.groupby(["rules", "agent"]).size(),
    "repos": per_agent_repo.groupby(["rules", "agent"]).size(),
})
by_agent.round(2)

repo_averaged_rate  prs  repos
rules           agent                                       
changelog       Claude_Code                 0.70   17      5
                Copilot                     0.48  226     15
                Cursor                      0.86   52      6
                Devin                       0.98   33      2
                Google_Jules                0.29    7      1
                OpenAI_Codex                0.15  179      7
commit_format   Claude_Code                 0.05  210     39
                Copilot                     0.34  734     77
                Cursor                      0.29  275     26
                Devin                       0.31  184      5
                OpenAI_Codex                0.10  640     47
package_manager Claude_Code                 0.00   34     15
                Copilot                     0.26   80     31
                Cursor                      0.16   66     15
                Devin                       0.00   74      7
                Google_Jules                0.00    1      1
                OpenAI_Codex                0.14   12      7
required_tests  Claude_Code                 0.21  139     28
                Copilot                     0.23  647     77
                Cursor                      0.41   92     15
                Devin                       0.80   41      4
                Google_Jules                1.00    1      1
                OpenAI_Codex                0.34  130     17

## 9. Hand check: package_manager violations

Open each PR link, look at the added lockfile, and fill the `correct` column in the CSV with `yes` / `no`.

In [16]:
def rule_line(hashes):
    for line in context_text(hashes).splitlines():
        if re.search(RULE_PATTERNS["package_manager"], line, re.IGNORECASE):
            return line.strip()


violations = pr_rule[(pr_rule.rules == "package_manager") & (pr_rule.label == "violated")]

to_check = pd.DataFrame({
    "pr_url": violations.html_url,
    "agent": violations.agent,
    "rule_line": violations.context_hashes.apply(rule_line),
    "required": violations.context_hashes.apply(lambda h: sorted(required_managers(context_text(h)))),
    "added_lockfiles": violations.added_files.apply(lambda files: [f for f in files if f.split("/")[-1] in LOCKFILE_MANAGER]),
    "correct": "",
})
to_check.to_csv("../../package_manager_violations.csv", index=False)

pd.set_option("display.max_colwidth", 120)
to_check

,pr_url,agent,rule_line,required,added_lockfiles,correct
700,https://github.com/ecomfe/vue-echarts/pull/923,Copilot,Use `pnpm install` to set up dependencies. `pnpm dev` serves the demo playground at `http://localhost:5173` for inte...,[pnpm],[package-lock.json],
2216,https://github.com/feldera/feldera/pull/4546,Copilot,- **Package Manager**: Bun (not npm),[bun],[web-console/package-lock.json],
2254,https://github.com/jesec/flood/pull/933,Copilot,# Package Manager: pnpm (v9.7.0) is the project's package manager,[pnpm],[package-lock.json],
2577,https://github.com/microsoft/typespec/pull/7661,Copilot,- Use `pnpm` as the package manager,[pnpm],[packages/http-client-java/package-lock.json],
3483,https://github.com/ast-grep/ast-grep-vscode/pull/495,Copilot,- **Node.js** and **pnpm**: Package manager for dependencies,[pnpm],[package-lock.json],
5674,https://github.com/demokratie-live/democracy-client/pull/1652,Copilot,- Yarn package manager (v1.22.19+),[yarn],[pnpm-lock.yaml],
6395,https://github.com/usememos/memos/pull/4814,Copilot,- **Package Manager:** pnpm,[pnpm],[web/package-lock.json],
7495,https://github.com/svelteplot/svelteplot/pull/182,Copilot,- **PNPM**: Package manager (preferred over npm/yarn),[pnpm],[package-lock.json],
7777,https://github.com/Azure/azure-sdk-for-js/pull/35787,Copilot,- Use pnpm for package management (never npm or yarn),[pnpm],[eng/common/tsp-client/package-lock.json],
8579,https://github.com/Fraunhofer-AISEC/cpg/pull/2386,Copilot,- **Always use `pnpm` for Node.js/JavaScript/TypeScript projects**,[pnpm],[codyze-console/src/main/webapp/package-lock.json],
